# TFM 3 · Causilo classifier suite (GPU)
Every way of turning Causilo into a die-yield count predictor, one step ahead on every row, several seeds. Scored with
coverage, CRPS, log score, log score − entropy, overdispersion. Needs the frequency-weights Causilo build (PR #19).
Predictions go to `results/tfm_bench/clf_suite/` as they're made; rerun to resume.

In [ ]:
# ---- settings ------------------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USER/ASML.git"   # <- your repo (Colab only; locally the notebook runs in place)
DRIVE_ROOT = "/content/drive/MyDrive/ASML"           # Colab only: Drive folder holding data/ and results/
STATE_ZIP = "../results/tfm_bench/benchmark_state_slim.zip"   # only used to get the data CSVs if data/ has none
SETTINGS = dict(
    DATASETS="auto:3",       # "auto:N" = N datasets with the most 0% / middle / 100% rows, "all", or a list of file names
    SEEDS=[0, 1, 2, 3, 4],
    MIN_CTX=40,              # first predicted row
    QUERY_STRIDE=1,          # 2 = every other row (half the cost)
    METHODS=["trials", "trials_bb", "calib", "calib_bb", "no_mid", "multiclass_71", "multiclass_4bin", "regressor"],
    TIME_BUDGET_H=11.0,
)

In [ ]:
# Setup. Works in Colab (clones the repo, links data/ + results/ to Drive) and in local Jupyter (runs in place).
# Every path below is relative to the notebooks/ folder:  ../data   ../results/tfm_bench   tfm_bench/*.py
import os, sys, glob, shutil, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules


def secret(name):
    """Colab secret (key icon on the left) or, locally, an environment variable. None if missing."""
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/ASML"
    tok = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    if os.path.isdir(f"{REPO}/.git"):   # the Colab copy is disposable: make it match GitHub exactly
        subprocess.run(["git", "-C", REPO, "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "@{u}"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, REPO], check=True)
    with open(f"{REPO}/.git/info/exclude", "a") as fh:   # keep the Drive links out of git even if .gitignore says "data/"
        fh.write("\n/data\n/results\n")
    os.chdir(f"{REPO}/notebooks")
    for sub in ["data", "results"]:     # not in git: link them to Drive so ../data and ../results work
        os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
        if not os.path.islink(f"../{sub}"):
            if os.path.isdir(f"../{sub}"):
                shutil.rmtree(f"../{sub}")
            os.symlink(f"{DRIVE_ROOT}/{sub}", f"../{sub}")
else:
    assert os.path.isdir("tfm_bench"), "run this notebook from the repo's notebooks/ folder"

DATA, WORK = os.path.abspath("../data"), os.path.abspath("../results/tfm_bench")
os.makedirs(DATA, exist_ok=True); os.makedirs(WORK, exist_ok=True)


def restore_state(zip_path, work_dir, data_dir):
    """Unzip a state zip: data CSVs -> data_dir (only if missing), results*/ folders -> work_dir.
    Handles zips whose content sits under a folder (e.g. project/), like the old Kaggle ones."""
    tmp = os.path.abspath("../results/_state_unzip")
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zip_path).extractall(tmp)
    hits = glob.glob(f"{tmp}/**/results*/*/results_*/predictions.csv", recursive=True)
    root = min((h.rsplit("/", 4)[0] for h in hits), key=len) if hits else tmp
    for sub in sorted(os.path.basename(p) for p in glob.glob(f"{root}/results*") if os.path.isdir(p)):
        shutil.copytree(f"{root}/{sub}", f"{work_dir}/{sub}", dirs_exist_ok=True)
    if os.path.exists(f"{root}/context_selection.csv"):
        shutil.copy(f"{root}/context_selection.csv", work_dir)
    for p in glob.glob(f"{tmp}/**/opt_results_*.csv", recursive=True):
        if not glob.glob(f"{data_dir}/**/{os.path.basename(p)}", recursive=True):
            shutil.copy(p, data_dir)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"restored {os.path.basename(zip_path)}: {len(glob.glob(f'{work_dir}/results*/*/results_*/predictions.csv'))} "
          f"results, {len(glob.glob(f'{data_dir}/**/opt_results_*.csv', recursive=True))} data CSVs")


print("notebooks dir:", os.getcwd(), "| data:", DATA, "| results:", WORK)

if not glob.glob(f"{DATA}/**/opt_results_*.csv", recursive=True):
    restore_state(os.path.abspath(STATE_ZIP), WORK, DATA)
import json
os.environ["CLF_SETTINGS"] = json.dumps(SETTINGS)

In [ ]:
%pip install -q --force-reinstall --no-deps "git+https://github.com/nums-ai/causilo@06b3fd0642a29d355581d6894899a3b5839686df"
!python -c "import inspect, causilo; assert 'sample_weight' in inspect.signature(causilo.CausiloClassifier.fit).parameters; print('PR #19 build OK')"
!nvidia-smi -L

In [ ]:
!python tfm_bench/classifier_suite.py

In [ ]:
# rebuild the report only (no GPU), e.g. after a partial run
!python tfm_bench/classifier_suite.py --report

In [ ]:
page = f"{WORK}/clf_suite/classifier_suite.html"   # self-contained; open it in any browser
if IN_COLAB:
    from google.colab import files
    files.download(page)
else:
    print(page)